### Расчет ABC/XYZ-категорий продуктов

Ноутбук использует функции из `get_product_abc_xyz_analysis.py`, чтобы не дублировать бизнес-логику.

Минимальный ноутбук: импорт, запуск расчета.

In [4]:
from get_product_abc_xyz_analysis import run_pipeline

In [6]:
abc_xyz_analysis_result = run_pipeline()
abc_xyz_analysis_result

Calculation started...


,SKU,Margin,Variation,Category
0,TDWFP001,275282.28,23.91,AX
1,TDWFP003,177059.54,23.23,AX
2,TDWFP010,115975.35,14.31,AX
3,TDWFP024,62987.69,21.21,AX
4,TDW0910,68429.76,20.75,AX
5,TDWFP007,106192.80,19.87,AX
6,TD250060U,103721.76,9.87,AX
7,TDWFP008,117254.02,18.46,AX
8,TDW4680,70767.04,36.06,AY
9,TDWFP022,65070.18,29.16,AY


In [386]:
data = pd.read_csv('for_abc-xyz.csv', index_col = 'Unnamed: 0')
data['Day'] = pd.to_datetime(data['Day'])
data['Date_ordinal'] = data['Day'].map(pd.Timestamp.toordinal)

In [387]:
data

,Day,SKU,Sold,Price,Cost,Status,Date_ordinal
0,2025-12-31,TD125060U,7,49.75,43.82,1,739616
1,2025-12-31,TDWFP003,17,32.46,9.59,1,739616
2,2025-12-31,TD125030U,30,13.26,3.48,1,739616
3,2025-12-31,T172060U,9,17.61,7.17,1,739616
4,2025-12-31,TDWFP002,60,33.86,23.15,1,739616
...,...,...,...,...,...,...,...
3026,2025-10-01,TDW3670,5,18.07,15.40,1,739525
3027,2025-10-01,TD360060U,7,49.93,25.28,1,739525
3028,2025-10-01,TDWFP014,28,35.79,22.50,1,739525
3029,2025-10-01,TDW2300,6,12.90,9.83,1,739525


In [388]:
prod = []
for sku in data['SKU'].unique():
    product = data[data['SKU'] == sku]
    if len(product) >= 90:
        prod.append(product.reset_index(drop = True))

In [389]:
def repare_prod(df1, df1_col = 'Sold', rez_col = 'Predicted Sold'):
    df = df1.copy()
    model = LinearRegression()
    model.fit(df1[df1['Status'] == 1][['Date_ordinal']], df1[df1['Status'] == 1][df1_col])
    repare_ordinals = df1[df1['Status'] == 0]['Date_ordinal']
    predicted_sales = model.predict(pd.DataFrame({'Date_ordinal': repare_ordinals}))
    df['Sold'] = df['Sold'].astype('float32')
    df.loc[df['Status'] == 0, 'Sold'] = [round(float(x), 2) if float(x) >= 0 else 0 for x in predicted_sales]
    return df

In [390]:
for i in range(len(prod)):
    if len(prod[i][prod[i]['Status'] == 0]) > 0:
        prod[i] = repare_prod(prod[i])

In [334]:
total_margin = 0
for i in range(len(prod)):
    marg = round(((prod[i]['Price'] - prod[i]['Cost']) * prod[i]['Sold']).sum(), 2)
    total_margin += marg

In [350]:
margin = pd.DataFrame()
margin['SKU'] = [prod[i].loc[0, 'SKU'] for i in range(len(prod))]
margin['Margin'] = [round(((prod[i]['Price'] - prod[i]['Cost']) * prod[i]['Sold']).sum(), 2) for i in range(len(prod))]
margin['Share'] = round(margin['Margin'] / total_margin * 100, 2)
margin = margin.sort_values(by = 'Share', ascending = False).reset_index(drop = True)
margin['Margin_cum'] = margin['Margin'].cumsum()
margin['Category'] = '-'
margin.loc[margin['Margin_cum'] < total_margin * 0.95, 'Category'] = 'B'
margin.loc[margin['Margin_cum'] < total_margin * 0.8, 'Category'] = 'A'
margin.loc[margin['Share'] < 3, 'Category'] = 'B'
margin.loc[margin['Margin_cum'] >= total_margin * 0.95, 'Category'] = 'C'
margin.loc[margin['Share'] <= 0, 'Category'] = 'D'
margin

,SKU,Margin,Share,Margin_cum,Category
0,TDWFP001,275282.28,17.76,275282.28,A
1,TDWFP003,177059.54,11.43,452341.82,A
2,TDWFP008,117254.02,7.57,569595.84,A
3,TDWFP010,115975.35,7.48,685571.19,A
4,TDWFP007,106192.80,6.85,791763.99,A
5,TD250060U,103721.76,6.69,895485.75,A
6,TDWFP002,73941.84,4.77,969427.59,A
7,TDW4680,70767.04,4.57,1040194.63,A
8,TDW0910,68429.76,4.42,1108624.39,A
9,TDWFP022,65070.18,4.20,1173694.57,A


In [351]:
stability = pd.DataFrame()
stability['SKU'] = [prod[i].loc[0, 'SKU'] for i in range(len(prod))]
stability['Avg_sold'] = [round(prod[i]['Sold'].sum() / len(prod[i]), 2) for i in range(len(prod))]
stability['St_deviation'] = [(((prod[i]['Sold'] - stability.loc[i, 'Avg_sold'])**2).sum() / len(prod[i]))**(1/2) for i in range(len(prod))]
stability['Coeff_variation'] = stability['St_deviation'] / stability['Avg_sold'] * 100
stability['Category'] = 'W'
stability.loc[stability['Coeff_variation'] > 0, 'Category'] = 'X'
stability.loc[stability['Coeff_variation'] > 25, 'Category'] = 'Y'
stability.loc[stability['Coeff_variation'] > 50, 'Category'] = 'Z'
stability.loc[stability['Coeff_variation'] > 100, 'Category'] = 'W'
stability

,SKU,Avg_sold,St_deviation,Coeff_variation,Category
0,TD125060U,8.28,9.065374,109.485195,W
1,TDWFP003,84.15,90.329341,107.343246,W
2,TD125030U,17.37,10.227539,58.880477,Z
3,T172060U,9.64,10.082897,104.594370,W
4,TDWFP002,75.04,84.121407,112.102089,W
5,TDWFP006,21.51,19.221532,89.360909,Z
6,TDW2300,8.28,6.420630,77.543838,Z
7,TD250060U,56.17,34.932111,62.189979,Z
8,TDWFP008,38.60,37.114412,96.151326,Z
9,TDWFP021,19.92,22.945699,115.189251,W


In [391]:
for i in range(len(prod)):
    prod[i]['Day'] = [day[:7] for day in prod[i]['Day'].astype('str')]

In [392]:
prod_months = []
for i in range(len(prod)):
    product = pd.DataFrame()
    product['Month'] = prod[i]['Day'].unique()
    product['SKU'] = prod[i].loc[0, 'SKU']
    product['Sold'] = [prod[i].loc[prod[i]['Day'] == month, 'Sold'].sum() for month in product['Month']]
    prod_months.append(product)

prod = prod_months

In [394]:
stability = pd.DataFrame()
stability['SKU'] = [prod[i].loc[0, 'SKU'] for i in range(len(prod))]
stability['Avg_sold'] = [round(prod[i]['Sold'].sum() / len(prod[i]), 2) for i in range(len(prod))]
stability['St_deviation'] = [(((prod[i]['Sold'] - stability.loc[i, 'Avg_sold'])**2).sum() / len(prod[i]))**(1/2) for i in range(len(prod))]
stability['Coeff_variation'] = stability['St_deviation'] / stability['Avg_sold'] * 100
stability['Category'] = 'W'
stability.loc[stability['Coeff_variation'] > 0, 'Category'] = 'X'
stability.loc[stability['Coeff_variation'] > 25, 'Category'] = 'Y'
stability.loc[stability['Coeff_variation'] > 50, 'Category'] = 'Z'
stability.loc[stability['Coeff_variation'] > 100, 'Category'] = 'W'
stability.sort_values(by = 'Coeff_variation').reset_index(drop = True)

,SKU,Avg_sold,St_deviation,Coeff_variation,Category
0,TD250060U,1722.67,170.082986,9.873219,X
1,TD527030U,342.00,38.218669,11.175049,X
2,TD125030U,532.67,62.622325,11.756308,X
3,TDW2300,254.00,34.669872,13.649556,X
4,TD360060U,233.00,32.934278,14.134883,X
5,TDWFP010,2962.33,423.890185,14.309351,X
6,TD440060U,516.33,76.877970,14.889309,X
7,TDWFP006,659.67,99.385892,15.066002,X
8,TDW2075,560.67,89.834416,16.022690,X
9,TDWFP008,1183.67,218.492309,18.458887,X


In [396]:
prod_list = [prod[i].loc[0, 'SKU'] for i in range(len(prod))]

In [417]:
result_matrix = pd.DataFrame()
result_matrix['SKU'] = prod_list
result_matrix['Margin'] = [list(margin[margin['SKU'] == sku]['Margin'])[0] for sku in prod_list]
result_matrix['Variation'] = [list(stability[stability['SKU'] == sku]['Coeff_variation'])[0] for sku in prod_list]
result_matrix['Category'] = [list(margin[margin['SKU'] == sku]['Category'])[0] + list(stability[stability['SKU'] == sku]['Category'])[0] for sku in prod_list]
result_matrix.sort_values(by = 'Category').reset_index(drop = True)

,SKU,Margin,Variation,Category
0,TDWFP001,275282.28,23.913385,AX
1,TDWFP003,177059.54,23.225100,AX
2,TDWFP010,115975.35,14.309351,AX
3,TDWFP024,62987.69,21.213666,AX
4,TDW0910,68429.76,20.749744,AX
5,TDWFP007,106192.80,19.872753,AX
6,TD250060U,103721.76,9.873219,AX
7,TDWFP008,117254.02,18.458887,AX
8,TDW4680,70767.04,36.057607,AY
9,TDWFP022,65070.18,29.161538,AY


In [87]:
tier_prices = pd.DataFrame()
tier_prices['From'] = [1, 20001, 50001]
tier_prices['To'] = [20000, 50000, -1]
tier_prices['Price'] = [1, 0.9, 0.7]

In [88]:
tier_prices

,From,To,Price
0,1,20000,1.0
1,20001,50000,0.9
2,50001,-1,0.7


In [107]:
price = 0
qty = 57628

for i in range(len(tier_prices)):
    if qty >= tier_prices.iloc[i]['From']:
        if qty >= tier_prices.iloc[i]['To'] and tier_prices.iloc[i]['To'] != -1:
            price += tier_prices.iloc[i]['Price'] * (tier_prices.iloc[i]['To'] - tier_prices.iloc[i]['From'] + 1)
        else: 
            price += tier_prices.iloc[i]['Price'] * (qty - tier_prices.iloc[i]['From'] + 1)

price

52339.6

In [424]:
import random
from sklearn.linear_model import LinearRegression
from datetime import timedelta

In [411]:
def fill_date(df):
    df['Day'] = pd.to_datetime(df['Day'], errors='coerce')  # Преобразуем в datetime
    full_range = pd.date_range(start=df['Day'].min(), end=df['Day'].max(), freq='D')
    df = df.set_index('Day')
    df = df.reindex(full_range, fill_value=0)
    return df.rename_axis('Day').reset_index()

In [412]:
def predict_sales(days_ahead, df1, df1_col = 'Sold', rez_col = 'Predicted Sold'):
    model = LinearRegression()
    model.fit(df1[['Date_ordinal']], df1[df1_col])
    last_date = df1['Day'].max()
    # future_dates = [last_date + timedelta(days=i) for i in range(1, days_ahead + 1)]
    future_dates = [pd.to_datetime(last_date) + timedelta(days=i) for i in range(1, days_ahead + 1)]
    future_ordinals = [d.toordinal() for d in future_dates]
    predicted_sales = model.predict(pd.DataFrame({'Date_ordinal': future_ordinals}))
    predicted_sales = [float(x) if float(x) >= 0 else 0 for x in predicted_sales]
    return pd.DataFrame({'Day': future_dates, rez_col: predicted_sales})

In [413]:
def prepare_prod(filename):
    data = pd.read_csv(filename)
    data['Day'] = pd.to_datetime(data['Day'])
    prod_list = list(data['Product variant SKU at time of sale'].unique())
    prod = [data[data['Product variant SKU at time of sale'] == sku].reset_index(drop = True) for sku in prod_list]

    for i in range(len(prod)):
        prod[i] = prod[i].sort_values(['Day'], ascending = False)
        term_prod = pd.DataFrame()
        term_prod['Day'] = prod[i]['Day'].unique()
        sold = []
        for date in list(prod[i]['Day'].unique()):
            sold.append(prod[i][prod[i]['Day'] == date]['Net items sold'].sum())
        term_prod['Sold'] = sold
        prod[i] = term_prod
        prod[i] = fill_date(prod[i])
        prod[i]['Date_ordinal'] = prod[i]['Day'].map(pd.Timestamp.toordinal)

    term_prod = []
    term_list = []
    for i in range(len(prod)):
        if len(prod[i]) >= 90:
            term_prod.append(prod[i])
            term_list.append(prod_list[i])
    return term_prod, term_list

In [414]:
def get_stocks(pred, dates, prod_list, filename):
    stocks = pd.concat([-pred[i]['Predicted Sold Total'] for i in range(len(pred))], axis = 1).transpose().reset_index(drop = True)
    stocks.columns = dates[1:]
    stocks[dates[0]] = pd.read_csv(filename)[dates[0]]
    stocks['SKU'] = prod_list
    for i in range(1, len(dates)):
        stocks[dates[i]] += stocks[dates[0]]
    return stocks

In [415]:
def include_PO(stocks_df, PO_filename, dates, prod_list):
    PO = pd.read_csv(PO_filename)
    for i in range(len(PO)):
        for j in range(dates.index(PO['Day'].iloc[i]), len(dates)):
            stocks_df.loc[prod_list.index(PO['SKU'].iloc[i]), dates[j]] += PO['Qty'].iloc[i]
    return stocks_df

In [416]:
def get_available_warehouse_space(stocks_df, dates, warehouse):
    warehouse_available = pd.DataFrame()
    warehouse_available['Day'] = dates
    warehouse_available['Space'] = [int(warehouse - stocks_df[dates[i]].sum()) for i in range(len(dates))]
    return warehouse_available

In [423]:
warehouse = 150000
date = '2025/31/12'
date = date[:4] + '-' + date[8:10] + '-' + date[5:7]
prod, prod_list = prepare_prod('sales_by_' + date + '.csv')
pred = []
for i in range(len(prod)):
    pred.append(predict_sales(1096, prod[i]))
    pred[i]['Predicted Sold Total'] = pred[i]['Predicted Sold'].cumsum()
dates = list(day[:4] + '/' + day[8:10] +'/' + day[5:7] for day in pd.date_range(start = str(prod[0]['Day'].max())[:10], periods = 1097).astype(str))

stocks = get_stocks(pred, dates, prod_list, 'inventory_level_on_' + date + '.csv')
stocks = include_PO(stocks, 'supplied_products_by_' + date + '.csv', dates, prod_list)
availiable_space = get_available_warehouse_space(stocks, dates, warehouse)
availiable_space

,Day,Space
0,2025/31/12,30671
1,2026/01/01,31707
2,2026/02/01,32745
3,2026/03/01,33784
4,2026/04/01,34823
...,...,...
1092,2028/27/12,2066370
1093,2028/28/12,2069241
1094,2028/29/12,2072114
1095,2028/30/12,2074989
